In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)
library(SeuratDisk)
library(clustree)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELmar26/"
repDir <- paste0(mainDir, "QC_clustering/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")

dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))
mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")
corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")
getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
mat <- read.delim(paste0(mainDir, "GSE57872_GBM_data_matrix.txt"), row.names = 1, check.names = FALSE)

# Convert to matrix
mat <- as.matrix(mat)

seu <- CreateSeuratObject(counts = mat, project = "Verhaak2021")

In [ ]:
### Load data from cellranger
samDir <- "/data/ebaird/scRNAseq/SCENTINELmar26/cellranger/scentinelmar26/outs/per_sample_outs/"
sams <- dir(samDir)
sams <- sams[sams %in% c("Eip93F", "HmgZ", "pros", "Syp")]
samDir <- paste0(samDir, sams, "/count/sample_filtered_feature_bc_matrix/")
names(samDir) <- sams

barcode.path <- paste0(samDir, "barcodes.tsv.gz")
features.path <- paste0(samDir, "features.tsv.gz")
matrix.path <- paste0(samDir, "matrix.mtx.gz")
file.exists(barcode.path)

mats <- lapply(1:length(barcode.path), function(i){
    mat <- readMM(file = matrix.path[i])
    gene.names = read.delim(features.path[i],
        header = FALSE,
        stringsAsFactors = FALSE)
    barcode.names = read.delim(barcode.path[i],
        header = FALSE,
        stringsAsFactors = FALSE)
    colnames(mat) = barcode.names$V1
    rownames(mat) = gene.names$V2
    mat
})
names(mats) <- names(samDir)

In [ ]:
### Check QC metrics
filters <- list(6000, 400, 500)
names(filters) <- c("maxGene", "minGene", "minUMI")

ncs <- lapply(mats, function(mat){
    cs <- colSums(mat)
    cg <- colSums(mat > 0)
    sapply(seq(450, to=10000, by=250), function(mu) sum(cg < filters$maxGene & cg > filters$minGene & cs > mu))
})

pdf(paste0(figDir, "cell.passing.filters.UMIs.pdf"))
plot(seq(450, to=10000, by=250), ncs[[1]], type="b", col=1, ylab="Number of cells passing filter", xlab="Filter on total number of UMIs per cell", ylim=c(0,12000))
for(i in 2:length(mats)) 
    lines(seq(450, to=10000, by=250), ncs[[i]], type="b", col=i, ylab="Number of cells passing filter", xlab="Filter on total number of UMIs per cell")
legend("topright", fill=1:4, legend=names(ncs))
dev.off()

ngs <- lapply(mats, function(mat){
    cs <- colSums(mat)
    cg <- colSums(mat > 0)
    sapply(seq(400, to=2000, by=50), function(mu) sum(cg < filters$maxGene & cg > mu & cs > filters$minUMI))
})

pdf(paste0(figDir, "cell.passing.filters.GENEs.pdf"))
plot(seq(400, to=2000, by=50), ngs[[1]], type="b", col=1, ylab="Number of cells passing filter", xlab="Filter on total number of genes per cell", ylim=c(0,12000))
for(i in 2:length(mats)) 
    lines(seq(400, to=2000, by=50), ngs[[i]], type="b", col=i)
legend("topright", fill=1:4, legend=names(ngs))
dev.off()

dfs <- lapply(names(mats), function(i){
    mat <- mats[[i]]
    data.frame(sample=i, umis=colSums(mat), ngenes=colSums(mat > 0))
})
dfs <- do.call(rbind, dfs)

jpeg(paste0(figDir, "umis.vs.ngenes.jpeg"), width=1500, height=1000)
ggplot(aes(umis, ngenes), data=dfs) +  geom_bin2d(bins = 1000) + facet_wrap(.~sample)
dev.off()

pdf(paste0(figDir, "dens_umis.pdf"), width=15, height=10)
ggplot(aes(umis, fill=sample), data=dfs) + geom_density(alpha=0.5) + coord_trans(x = "log10") + geom_vline(xintercept=700)
ggplot(aes(ngenes, fill=sample), data=dfs) + geom_density(alpha=0.5) + coord_trans(x = "log10") + geom_vline(xintercept=300)
dev.off()

print(ggplot(aes(umis, ngenes), data=dfs) +  geom_bin2d(bins = 1000) + facet_wrap(.~sample))
print(ggplot(aes(umis, fill=sample), data=dfs) + geom_density(alpha=0.5) + coord_trans(x = "log10") + geom_vline(xintercept=700))
print(ggplot(aes(ngenes, fill=sample), data=dfs) + geom_density(alpha=0.5) + coord_trans(x = "log10") + geom_vline(xintercept=300))

sink(paste0(tabDir, "num_cells_pass.txt"))
table(dfs$umis > 700 & dfs$ngenes > 300, dfs$sample)
sink()

In [ ]:
## Create merged seurat object
seus <- lapply(mats, function(mat) CreateSeuratObject(counts=mat))
for(i in names(seus)) seus[[i]]$sample <- i
for(i in names(seus)) seus[[i]]$orig.ident <- NULL
seu <- merge(seus[[1]], seus[2:4])
seu[["percent.ribo"]] <- PercentageFeatureSet(seu, pattern = "RpS|RpL")
seu[["percent.mt"]] <- PercentageFeatureSet(seu, pattern = "^mt:")

### Add metadata
geno <- c("Eip93F", "HmgZ", "pros", "Syp")
names(geno) <- c("Eip93F", "HmgZ", "pros", "Syp")
geno <- geno[seu$sample]
names(geno) <- colnames(seu)
seu$genotype <- geno
# timepoint <- c("12d", "10d", "10d", "12d")
# names(timepoint) <- c("12d", "10d", "10d", "12d")
# timepoint <- timepoint[seu$sample]
# names(timepoint) <- colnames(seu)
# seu$timepoint <- timepoint
seu$condition <- seu$sample

plot <- VlnPlot(seu, features = c("nFeature_RNA", "nCount_RNA", "percent.mt", "percent.ribo"), ncol = 4, pt.size=0, group.by = "sample")
pdf(file.path(figDir, "vln_plot_QC_unfiltered.pdf"))
print(plot)
dev.off()
print(plot)

In [ ]:
### Normalise and scale data
seu <- SCTransform(seu, verbose = FALSE, return.only.var.genes = FALSE, conserve.memory = TRUE, vst.flavor = "v1")
seu <- RunPCA(seu, verbose = FALSE, npcs = 50)

pdf(paste0(figDir, "PCA.elbow.unfiltered.pdf"))
ElbowPlot(seu, ndims=50)
dev.off()
print(ElbowPlot(seu, ndims=50))

In [ ]:
### RAW umap and clusters
seu <- RunUMAP(seu, dims = 1:16)
seu <- FindNeighbors(seu, dims = 1:16, verbose = FALSE)
seu <- FindClusters(seu, verbose = FALSE, resolution=1)

jpeg(paste0(figDir, "UMAP.unfiltered.jpeg"),width = 1800, height = 1800, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)

In [ ]:
### Unfiltered QC metrics
plot <- VlnPlot(seu, features = c("nFeature_RNA", "nCount_RNA", "percent.mt", "percent.ribo"), ncol = 4, pt.size=0, group.by = "condition")
pdf(file.path(figDir, "vln_plot_QC_unfiltered.b.pdf"))
print(plot)
dev.off()
print(plot)

In [ ]:
### Compute cell cycle
all_cell_cycle <- read.table("/data/ebaird/scRNAseq/SCENTINELsep24/refs/Dm_cell_cycle_genes.csv", sep=",", header=T)
all_cell_cycle <- all_cell_cycle[!is.na(all_cell_cycle$geneID), ]
g2 <- all_cell_cycle[all_cell_cycle$phase=="G2/M",]$geneID
s <- all_cell_cycle[all_cell_cycle$phase=="S",]$geneID

seu <- CellCycleScoring(seu, s.features = s, g2m.features = g2, set.ident = FALSE)

### Doublets part 1
seuf <- SplitObject(seu, split.by="condition")

seuf <- lapply(seuf, function(x) {
    x <- SCTransform(x, verbose = FALSE, return.only.var.genes = FALSE)
    RunPCA(x, verbose = FALSE)
})

pdf(paste0(figDir, "PCA.elbows.doublets.pdf"))
for(i in names(seuf)) print(ElbowPlot(seuf[[i]], ndims=30))
dev.off()
for(i in names(seuf)) print(ElbowPlot(seuf[[i]], ndims=30))

In [ ]:
### Doublets part 2
seuf <- lapply(seuf, function(x) {
    x <- RunUMAP(x, dims = 1:13)
    x <- FindNeighbors(x, dims = 1:13, verbose = FALSE)
    FindClusters(x, verbose = FALSE, resolution=1)
})

sce <- lapply(seuf, function(x) as.SingleCellExperiment(x))

dbls <- lapply(sce, function(sc) scDblFinder(sc, clusters=NULL, BPPARAM=BiocParallel::MulticoreParam(5)))

for(i in names(dbls)) seuf[[i]]$scDblFinder.class <- dbls[[i]]$scDblFinder.class

mds <- lapply(seuf, function(x) x@meta.data)
for(i in 1:2) rownames(mds[[i]]) <- paste0(rownames(mds[[i]]), "_", i)
mds <- do.call(rbind, unname(mds))

seu$scDblFinder.class <- mds[colnames(seu),]$scDblFinder.class

library(reshape2)
ta <- melt(table(seu$scDblFinder.class, seu$seurat_clusters))

ta <- data.frame(ta %>% group_by(Var2) %>% summarise(prop=value[2]/sum(value), tot=sum(value)))

seu$dblsClus <- ta$prop[match(seu$seurat_clusters, ta$Var2)]

In [ ]:
### Save/load unfiltered object
# saveRDS(seu, paste0(repDir, "unfiltered.rds"))
seu <- readRDS(paste0(repDir, "unfiltered.rds"))

In [ ]:
### RAW QC feature plots
gs <- lapply(c( "nCount_RNA", "nFeature_RNA", "RFP", "percent.mt", "percent.ribo", "S.Score", "G2M.Score"), function(i) FeaturePlot(seu, features=i, order=T, ncol=3, max.cutoff="q99") & scale_colour_gradientn(colours = cols))
gs[[length(gs)+1]] <- DimPlot(seu, group.by="Phase")
gs[[length(gs)+1]] <- DimPlot(seu, group.by="scDblFinder.class")
lay <- rbind(c(1,2,3),
             c(4,5,6),
             c(7,8,9))            
jpeg_file <- paste0(figDir, "UMAP.QC.unfiltered.jpeg")
jpeg(filename = jpeg_file, width = 1800, height = 1800, res = 150)
gridExtra::grid.arrange(grobs = gs, layout_matrix = lay)
dev.off()

# Calculate the summary statistics for nFeature, nCount and percent.mt across clusters
ncount_stats <- seu@meta.data %>%
  group_by(seurat_clusters) %>%
  summarize(
    mean_nCount = mean(nCount_RNA),
    median_nCount = median(nCount_RNA),
    min_nCount = min(nCount_RNA),
    max_nCount = max(nCount_RNA),
    sd_nCount = sd(nCount_RNA),
    n_cells = n()
  ) %>%
  ungroup()

nfeature_stats <- seu@meta.data %>%
  group_by(seurat_clusters) %>%
  summarize(
    mean_nFeature = mean(nFeature_RNA),
    median_nFeature = median(nFeature_RNA),
    min_nFeature = min(nFeature_RNA),
    max_nFeature = max(nFeature_RNA),
    sd_nFeature = sd(nFeature_RNA),
    n_cells = n()
  ) %>%
  ungroup()

percent_mt_stats <- seu@meta.data %>%
  group_by(seurat_clusters) %>%
  summarize(
    mean_percent_mt = mean(percent.mt),
    median_percent_mt = median(percent.mt),
    min_percent_mt = min(percent.mt),
    max_percent_mt = max(percent.mt),
    sd_percent_mt = sd(percent.mt),
    n_cells = n()
  ) %>%
  ungroup()

summary_stats <- left_join(ncount_stats, nfeature_stats, by = "seurat_clusters", suffix = c("_nCount", "_nFeature"))

summary_stats <- summary_stats %>%
  left_join(percent_mt_stats, by = "seurat_clusters")

sink(paste0(tabDir, "ncount_nfeature_percent.mt_stats.txt"))
summary_stats
sink()

# Facet plots for nFeature, nCount and percent.mt per cluster
p1 <- ggplot(seu@meta.data, aes(x = factor(seurat_clusters), y = nCount_RNA)) +
  geom_violin(fill = "skyblue", color = "black") +
  facet_wrap(~ seurat_clusters, scales = "free") +
  labs(title = "Distribution of nCount_RNA across Clusters", 
       x = "Cluster", y = "nCount_RNA") +
  theme_minimal()

p2 <- ggplot(seu@meta.data, aes(x = factor(seurat_clusters), y = nFeature_RNA)) +
  geom_violin(fill = "lightgreen", color = "black") +
  facet_wrap(~ seurat_clusters, scales = "free") +
  labs(title = "Distribution of nFeature_RNA across Clusters", 
       x = "Cluster", y = "nFeature_RNA") +
  theme_minimal()

p3 <- ggplot(seu@meta.data, aes(x = factor(seurat_clusters), y = percent.mt)) +
  geom_violin(fill = "lightgreen", color = "black") +
  facet_wrap(~ seurat_clusters, scales = "free") +
  labs(title = "Distribution of percent.mt across Clusters", 
       x = "Cluster", y = "percent.mt") +
  theme_minimal()

pdf(file = paste0(figDir, "vln.plots.nfeature.ncount.percentmt.per.cluster.pdf"), width = 15, height = 10)
print(p1)
print(p2)
print(p3)
dev.off()

print(p1)
print(p2)
print(p3)

In [ ]:
### Toggle filters to visualise thresholds on UMAP before applying
ncount_lower_threshold <- 1300
ncount_upper_threshold <- 20000

nfeature_lower_threshold <- 500
nfeature_upper_threshold <- 3000

percent_mt_lower_threshold <- 1
percent_mt_upper_threshold <- 15

seu$nCount_pass <- ifelse(seu$nCount_RNA >= ncount_lower_threshold & seu$nCount_RNA <= ncount_upper_threshold, 1, 0)
seu$nFeature_pass <- ifelse(seu$nFeature_RNA >= nfeature_lower_threshold & seu$nFeature_RNA <= nfeature_upper_threshold, 1, 0)
seu$percent_mt_pass <- ifelse(seu$percent.mt >= percent_mt_lower_threshold & seu$percent.mt <= percent_mt_upper_threshold, 1, 0)

gs <- lapply(c("nCount_pass", "nFeature_pass", "percent_mt_pass"), function(i) {
  FeaturePlot(seu, features = i, order = TRUE, ncol = 1, max.cutoff = "q99", reduction = "umap") +
    scale_colour_gradientn(colours = c("blue", "gray")) + theme(legend.position = "none")
})

gs[[length(gs) + 1]] <- DimPlot(seu, group.by = "SCT_snn_res.1", label = TRUE, label.size = 3, reduction = "umap") + theme(legend.position = "none")

print(gs)

lay <- rbind(c(1, 2), 
             c(3, 4))

jpeg_file <- paste0(figDir, "UMAP.filter.thresholds.jpeg")
jpeg(filename = jpeg_file, width = 1800, height = 1800, res = 150)
gridExtra::grid.arrange(grobs = gs, layout_matrix = lay)
dev.off()

In [ ]:
### Apply the filtering to subset the data based on thresholds
seu <- subset(seu, cells=Cells(seu)[seu$nCount_RNA > 1300 & seu$nCount_RNA < 20000 & seu$nFeature_RNA > 500 & seu$nFeature_RNA < 3000 & seu$percent.mt < 15 & seu$percent.mt > 1])
ribosomal_genes <- grep("^Rp[SL]", rownames(seu), value = TRUE)
seu <- seu[!rownames(seu) %in% ribosomal_genes, ]
 
plot <- VlnPlot(seu, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3, pt.size=0, group.by = 'condition')
pdf(file.path(figDir, "vln_plot_QC_filtered.pdf"))
print(plot)
dev.off()
print(plot)

DefaultAssay(seu) <- "RNA"
seu <- JoinLayers(seu)
seu <-NormalizeData(seu)
genevar <- data.frame(modelGeneVar(seu@assays$RNA$data))
genevar$gene <- rownames(genevar)

pdf(paste0(figDir, "mean_var.pdf"), width=16)
ggplot(aes(mean,total),data=genevar)+geom_point(size=0.7)+scale_x_continuous(trans='log10') + scale_y_continuous(trans='log10')
dev.off()
print(ggplot(aes(mean,total),data=genevar)+geom_point(size=0.7)+scale_x_continuous(trans='log10') + scale_y_continuous(trans='log10'))

In [ ]:
### Remove doublets
seu <- subset(seu, cells=colnames(seu)[seu$scDblFinder.class != "doublet"])

In [ ]:
### Regress phase and mt.percent
seu <- SCTransform(seu, verbose = FALSE, vars.to.regress=c("S.Score", "G2M.Score", "percent.mt"), return.only.var.genes = FALSE, conserve.memory = TRUE, vst.flavor = "v1")
seu <- RunPCA(seu, verbose = FALSE)
pdf(paste0(figDir, "PCA_elbow_filtered_regressed.pdf"))
ElbowPlot(seu, ndims=30)
dev.off()
ElbowPlot(seu, ndims=30)

In [ ]:
# Plot UMAP at varying dimensions, fixed resolution at 1.0
dims_range <- 13:30
plots_dims <- list()

for (d in dims_range) {

  seu_tmp <- seu
  
  seu_tmp <- RunUMAP(seu_tmp, dims = 1:d, verbose = FALSE)
  seu_tmp <- FindNeighbors(seu_tmp, dims = 1:d, verbose = FALSE)
  seu_tmp <- FindClusters(seu_tmp, resolution = 1.0, verbose = FALSE)

  p <- DimPlot(
        seu_tmp,
        group.by = "SCT_snn_res.1",
        reduction = "umap",
        label = TRUE
      ) +
      ggtitle(paste0("dims = ", d, ", res = 1.0"))

  plots_dims[[length(plots_dims) + 1]] <- p
}

jpeg_file <- paste0(figDir, "UMAP_dims_sweep.jpeg")

jpeg(filename = jpeg_file, width = 4000, height = 4000, res = 200)

gridExtra::grid.arrange(
  grobs = plots_dims,
  ncol = 4
)

dev.off()

In [ ]:
# Plot UMAP at fixed dimensions, varying resolutions
dims_use <- 20
resolutions <- seq(0.6, 1.6, by = 0.1)

seu <- RunUMAP(seu, dims = 1:dims_use, verbose = FALSE)
seu <- FindNeighbors(seu, dims = 1:dims_use, verbose = FALSE)

seu <- FindClusters(
  seu,
  resolution = resolutions,
  verbose = FALSE
)

plots_res <- lapply(resolutions, function(r) {

  colname <- paste0("SCT_snn_res.", r)

  DimPlot(
    seu,
    group.by = colname,
    reduction = "umap",
    label = TRUE
  ) +
  ggtitle(paste0("dims = ", dims_use, ", res = ", r))

})

jpeg_file <- paste0(figDir, "UMAP_resolution_sweep.jpeg")

jpeg(filename = jpeg_file, width = 4000, height = 3000, res = 200)

gridExtra::grid.arrange(
  grobs = plots_res,
  ncol = 4
)

dev.off()

In [ ]:
clustree(seu, prefix = "SCT_snn_res.")

In [ ]:
### Recompute final UMAP and resolution and plot QC features
seu <- RunUMAP(seu, dims = 1:20)
seu <- FindNeighbors(seu, dims = 1:20, verbose = FALSE)
seu <- FindClusters(seu, verbose = FALSE, resolution=1.1)

DimPlot(seu, group.by = "SCT_snn_res.1.1", label = TRUE, label.size = 3, reduction = "umap")

gs <- lapply(c( "nCount_RNA", "nFeature_RNA", "RFP", "percent.mt", "percent.ribo", "S.Score", "G2M.Score"), function(i) FeaturePlot(seu, features=i, order=T, ncol=3, max.cutoff="q99") & scale_colour_gradientn(colours = cols))
gs[[length(gs)+1]] <- DimPlot(seu, group.by="SCT_snn_res.1.1", label=TRUE, label.size=3) + theme(legend.position = "none")
gs[[length(gs)+1]] <- DimPlot(seu, group.by="Phase")
lay <- rbind(c(1,2,3),
             c(4,5,6),
             c(7,8,9))
jpeg_file <- paste0(figDir, "UMAP.QC.filtered.regressed.jpeg")
jpeg(filename = jpeg_file, width = 1800, height = 1800, res = 150)
gridExtra::grid.arrange(grobs = gs, layout_matrix = lay)
dev.off()

In [ ]:
### Save UMAP after regression
jpeg(paste0(figDir, "UMAP.full.jpeg"), width = 900, height = 900, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5) +
  theme(legend.position = "none")
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)

jpeg(paste0(figDir, "UMAP.merged_clusters_split_by_cond.jpeg"), width = 1800, height = 1800, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "sample", ncol = 2) +
  theme(legend.position = "none")
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "sample", ncol = 2) +
  theme(legend.position = "none")

In [ ]:
### Save/load regressed object
saveRDS(seu, file=paste0(repDir, "filtered_regressed.rds"))
# seu <- readRDS(paste0(repDir, "filtered_regressed.rds"))

In [ ]:
## Find putative markers for clusters
DefaultAssay(seu)<-'SCT'
Idents(seu) <- paste0('seurat_clusters')
all.markers <- FindAllMarkers(seu, only.pos = TRUE, min.pct = 0.2, logfc.threshold = 0.5)
write.csv(all.markers,file=paste0(tabDir,'allMarkers.csv'))
all.markers %>%
        group_by(cluster) %>%
        slice_max(n = 5, order_by = avg_log2FC) -> top5
write.csv(top5,file=paste0(tabDir,'top5Markers.csv'))
#
pdf(paste0(figDir,'top5markers.heatmap.pdf'),width=10,height=15)
print(DoHeatmap(seu, features = top5$gene) + NoLegend() + theme(axis.text.y = element_text(size = 8)))
dev.off()

jpeg(paste0(figDir, 'top5markers.dotplot.jpeg'), quality = 100, width = 2500, height = 1000, res = 150)
print(
  DotPlot(seu, features = unique(top5$gene), dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 5)
  ) +
  scale_color_gradientn(
    colours = c("white", "forestgreen"),
    limits = c(0, 1.5),
    oob = scales::squish
  ) +
  ggtitle("Top 5 Markers per cluster")
)
dev.off()

In [ ]:
### DE between specific clusters that potentially need merging based on heatmap and dotplot visualisation

cluster_pairs <- list(
  "16_vs_18" = c("16", "18"),
  "16_vs_7" = c("16", "7"),
  "7_vs_18" = c("7", "18"),
  "1_vs_2"  = c("1", "2"),
  "5_vs_0"  = c("5", "0"),
  "7_vs_5"  = c("7", "5"),
  "7_vs_0"  = c("7", "0")
)

for (pair_name in names(cluster_pairs)) {
  clusters <- cluster_pairs[[pair_name]]
  
  cluster_subset <- subset(seu, idents = clusters)
  Idents(cluster_subset) <- "seurat_clusters"
  
  de_results <- FindMarkers(
    object = cluster_subset,
    ident.1 = clusters[1],
    ident.2 = clusters[2],
    min.pct = 0.25,
    logfc.threshold = 0.25
  )
  
  filtered_results <- subset(de_results, p_val < 0.05 & abs(avg_log2FC) > 0.5)
  write.csv(filtered_results, file.path(tabDir, paste0("DE_results_cluster_", pair_name, ".csv")))
  
  top_genes <- head(rownames(filtered_results[order(-abs(filtered_results$avg_log2FC)), ]), 6)
  
  plots <- list()
  for (gene in top_genes) {
    p <- VlnPlot(cluster_subset, features = gene, group.by = "seurat_clusters", pt.size = 0) +
      ggtitle(paste0("Cluster ", pair_name, " - ", gene)) + theme(plot.title = element_text(size = 10))
    plots[[gene]] <- p
  }
  
  plots <- wrap_plots(plots, ncol = 2)
  
  pdf(file.path(figDir, paste0("violin_plots_cluster_comparison_", pair_name, ".pdf")))
  print(plots)
  dev.off()
}

In [ ]:
# Merge clusters
new_cluster_ids <- c('0', '1', '1', '2', '3', '0', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '5', '14', '5', '15')
names(new_cluster_ids) <- levels(seu)
seu <- RenameIdents(seu, new_cluster_ids)
seu@meta.data$seurat_clusters <- Idents(seu)

In [ ]:
### Save UMAP with merged clusters
jpeg(paste0(figDir, "UMAP.merged_clusters_full.jpeg"), width = 900, height = 900, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5) +
  theme(legend.position = "none")
dev.off()
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5)

DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "genotype", ncol = 2) +
  theme(legend.position = "none")
jpeg(paste0(figDir, "UMAP.merged_clusters_split_by_cond.jpeg"), width = 1800, height = 1800, res = 150)
DimPlot(seu, reduction = "umap", group.by = "seurat_clusters", label = TRUE, label.size = 5, split.by = "genotype", ncol = 2) +
  theme(legend.position = "none")
dev.off()

In [ ]:
# Basic cell counts
cell_counts <- table(seu$genotype)
print(cell_counts)

# With percentages
cell_counts_percent <- prop.table(table(seu$genotype)) * 100
print(cell_counts_percent)

In [ ]:
### Keep cells in cluster 15 with at least one hemocyte marker
hemocyte_markers <- c("Hml",          "Pxn",          "crq",          "NimC1",          "NimB4",          "NimB5",          "Col4a1",          "PPO1",          "PPO2",          "lz",          "peb",          "PPO3",          "msn",          "mys",          "cher",          "regucalcin",          "SPARC",          "Npc2a",          "Ppn",          "Inos",          "Nplp2",          "CG14629",          "CG34331",          "srp",          "drpr"
)
cluster_15 <- subset(seu, idents = "15")
counts <- GetAssayData(cluster_15, slot = "counts")
present_markers <- hemocyte_markers[hemocyte_markers %in% rownames(counts)]
if (length(present_markers) == 0) {
  stop("None of the hemocyte markers were found in cluster 15.")
}
marker_expr <- counts[present_markers, , drop = TRUE]
expressed_cells <- colnames(marker_expr)[Matrix::colSums(marker_expr > 0) > 0]
cluster_15 <- subset(cluster_15, cells = expressed_cells)
counts <- GetAssayData(cluster_15, slot = "counts")

seu <- subset(seu, cells = c(Cells(seu)[seu$seurat_clusters != "15"], colnames(cluster_15)))

### save counts from cluster 15
write.csv(as.data.frame(counts), file = paste0(tabDir, "cluster 15 hemocyte marker counts.csv"))

In [ ]:
### Remove sex genes and transposable elements
sex_genes <- c("msl-1", 'msl-2', 'msl-3', 'mof', 'mle', 'lncRNA:roX1', 'lncRNA:roX2', 'lncRNA:noe')
transposons <- grep("\\{\\}", rownames(seu), value = TRUE)
genes_to_keep <- setdiff(rownames(seu), sex_genes)
genes_to_keep <- setdiff(genes_to_keep, transposons)

DefaultAssay(seu) <- "RNA"

seu_nosex <- CreateSeuratObject(
  counts = GetAssayData(seu, assay = "RNA", slot = "counts")[genes_to_keep, ],
  data = GetAssayData(seu, assay = "RNA", slot = "data")[genes_to_keep, ],
  meta.data = seu@meta.data
)

if ("SCT" %in% names(seu@assays)) {
  seu_nosex[["SCT"]] <- CreateAssayObject(
    data = GetAssayData(seu, assay = "SCT", slot = "data")[genes_to_keep, ]
  )
  
  seu_nosex <- SetAssayData(
    object = seu_nosex,
    assay = "SCT",
    slot = "counts",
    new.data = GetAssayData(seu, assay = "SCT", slot = "counts")[genes_to_keep, ]
  )
  
  if (!is.null(GetAssayData(seu, assay = "SCT", slot = "scale.data"))) {
    seu_nosex <- SetAssayData(
      object = seu_nosex,
      assay = "SCT",
      slot = "scale.data",
      new.data = GetAssayData(seu, assay = "SCT", slot = "scale.data")[genes_to_keep, ]
    )
  }
  
  VariableFeatures(seu_nosex, assay = "SCT") <- intersect(
    VariableFeatures(seu, assay = "SCT"),
    genes_to_keep
  )
}

seu_nosex@reductions <- seu@reductions
seu_nosex@graphs <- seu@graphs
seu <- seu_nosex

In [ ]:
### Save/load finalised clusters object
# saveRDS(seu, file=paste0(repDir, "final_clusters.rds"))
seu <- readRDS(paste0(repDir, "final_clusters.rds"))

In [ ]:
## Find markers for finalised clusters
# DefaultAssay(seu)<-'SCT'
# Idents(seu) <- paste0('seurat_clusters')
# all.markers <- FindAllMarkers(seu, only.pos = TRUE, min.pct = 0.2, logfc.threshold = 0.5)
# write.csv(all.markers,file=paste0(tabDir,'allMarkers_final_clusters.csv'))
# all.markers %>%
#         group_by(cluster) %>%
#         slice_max(n = 10, order_by = avg_log2FC) -> top
# write.csv(top,file=paste0(tabDir,'top10Markers_final_clusters.csv'))

# jpeg(paste0(figDir,'top10markers.heatmap_final_clusters.jpeg'), quality = 100, width = 2500, height = 1000, res = 150)
# print(DoHeatmap(seu, features = top$gene) + NoLegend())
# dev.off()
#
jpeg(paste0(figDir, 'top10markers.dotplot_final_clusters.jpeg'), quality = 100, width = 3800, height = 1000, res = 150)
print(
  DotPlot(seu, features = unique(top$gene), dot.scale = 6) + 
  RotatedAxis() +
  theme(
    axis.text.x = element_text(size = 7)
  ) +
  scale_color_gradientn(
    colours = c("white", "forestgreen"),
    limits = c(0, 1.5),
    oob = scales::squish
  ) +
  ggtitle("Top 10 Markers per cluster")
)
dev.off()

In [ ]:
### Cluster marker genes split by genotype
unique_genotypes <- unique(seu$genotype)
genes_to_plot <- unique(top5$gene)

plots <- list()
for (gen in unique_genotypes) {
  seu_subset <- subset(seu, subset = genotype == gen)
  p <- DotPlot(seu_subset, features = genes_to_plot, dot.scale = 6) +
    scale_color_gradientn(colours = c("white", "blue"), limits = c(0, 1.5), oob = scales::squish) +
    ggtitle(gen) +
    RotatedAxis() +
    theme(
      axis.text.x = element_text(size = 5),
      plot.title = element_text(hjust = 0.5)
    ) +
    guides(color = guide_colorbar(title = "Expr"),
           size = guide_legend(title = "Pct. Expr"))
  plots[[gen]] <- p
}

combined <- wrap_plots(plots, ncol = 1)

jpeg(paste0(figDir, 'top5markers_dotplot_per_genotype.jpeg'),
     quality = 100, width = 1800, height = 2000, res = 150)
print(combined)
dev.off()